# 텍스트 분석 종합과제 — 영화 흥행 예측

## 비즈니스 시나리오

> 여러분은 영화 데이터 분석 스타트업 **CineInsight** 의 주니어 데이터 분석가입니다. \
> 투자팀은 "어떤 영화가 흥행할 것인가?"를 예측하기 위해 \
> **영화 줄거리(overview) 텍스트와 장르 정보만으로 흥행 여부를 분류하는 모델** 을 요청했습니다. \
> TF-IDF 분석, 토픽 모델링, 딥러닝 분류 모델을 활용하여 분석 보고서를 작성해 주세요.

---

| Part | 주제 | 핵심 질문 |
|------|------|-----------|
| **Part 0** | 데이터 전처리 | (코드 제공) |
| **Part 1** | TF-IDF 분석 + 연관 분석 | Hit/Flop 영화의 텍스트 특징은 무엇인가? |
| **Part 2** | BERTopic 토픽 모델링 | 영화 줄거리에 숨겨진 토픽은 무엇인가? |
| **Part 3** | 텍스트 분류 모델 (BERT + 로지스틱 회귀) | 텍스트만으로 흥행 여부를 예측할 수 있는가? |

### 데이터셋 개요

| 항목 | 내용 |
|------|------|
| **파일명** | `data/movies_clean.csv` |
| **크기** | 약 9,536행 x 28열 |
| **도메인** | TMDB 영화 메타데이터 |

**주요 변수:**

| # | 변수명 | 설명 |
|---|--------|------|
| 1 | `title` | 영화 제목 |
| 2 | `overview` | 영화 줄거리 (영문) |
| 3 | `genres` | 장르 리스트 (JSON 형식) |
| 4 | `vote_average` | 평균 평점 (0~10) |
| 5 | `vote_count` | 투표 수 |
| 6 | `popularity` | 인기도 점수 |
| 7 | `budget` | 제작 예산 |
| 8 | `revenue` | 수익 |
| 9 | `year` | 개봉 연도 |
| 10 | `runtime` | 상영 시간 (분) |

In [ ]:
import json
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from wordcloud import WordCloud
import networkx as nx
import warnings
import platform
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS # 영어 불용어

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬러 팔레트
COLORS = {
    'hit': '#10B981',     # 에메랄드 (Hit)
    'flop': '#EF4444',    # 빨강 (Flop)
    'blue': '#3B82F6',
    'indigo': '#6366F1',
    'violet': '#8B5CF6',
    'teal': '#14B8A6',
    'amber': '#F59E0B',
    'slate': '#64748B',
    'sky': '#0EA5E9',
}


---
## Part 0: 데이터 전처리 (코드 제공)

아래 코드를 실행하여 분석에 사용할 데이터프레임 `df`를 준비합니다.

 ### 0-1. 데이터 로드 및 필터링

In [ ]:
df_raw = pd.read_csv('data/movies_clean.csv')
print(f"원본 데이터: {df_raw.shape[0]:,}행 x {df_raw.shape[1]}열")

# overview가 존재하고, vote_count >= 10 인 행만 사용 (평점 신뢰도 확보)
df = df_raw[df_raw['overview'].notna() & (df_raw['vote_count'] >= 10)].copy()
print(f"필터링 후: {len(df):,}행 (overview 존재 + vote_count >= 10)")


 ### 0-2. 파생변수 생성

In [ ]:
# 평점 기준으로 Hit/Flop 분류 (8점 이상 = Hit, 5점 이하 = Flop, 중간은 제외)
HIT_THRESHOLD = 8.0
FLOP_THRESHOLD = 5.0

print(f"분류 기준: Hit >= {HIT_THRESHOLD}, Flop <= {FLOP_THRESHOLD}")
print(f"  전체 데이터: {len(df):,}건")
print(f"  Hit (>= {HIT_THRESHOLD}): {(df['vote_average'] >= HIT_THRESHOLD).sum():,}건")
print(f"  Flop (<= {FLOP_THRESHOLD}): {(df['vote_average'] <= FLOP_THRESHOLD).sum():,}건")
print(f"  제외 ({FLOP_THRESHOLD} < vote < {HIT_THRESHOLD}): {((df['vote_average'] > FLOP_THRESHOLD) & (df['vote_average'] < HIT_THRESHOLD)).sum():,}건")

# 중간 평점(5~8) 제외 => 명확한 Hit/Flop만 사용
df = df[(df['vote_average'] >= HIT_THRESHOLD) | (df['vote_average'] <= FLOP_THRESHOLD)].copy()
df['label'] = (df['vote_average'] >= HIT_THRESHOLD).astype(int)
df['label_name'] = df['label'].map({1: 'Hit', 0: 'Flop'})

# genres 문자열 파싱 => 공백 구분 텍스트
df['genres_text'] = df['genres'].apply(lambda x: ' '.join(json.loads(x)) if pd.notna(x) else '')

df = df.reset_index(drop=True)

print()
print(f"Hit: {(df['label'] == 1).sum():,}건 / Flop: {(df['label'] == 0).sum():,}건")
print(f"분석 대상: {len(df):,}건")
print()
print("파생변수 생성 완료:")
for col in ['label', 'label_name', 'genres_text']:
    print(f"  - {col}: {df[col].dtype}")


### 0-3. 영어 텍스트 전처리

영어 텍스트 분석 파이프라인: **(1) 정제 => (2) 정규화 => (3) 토큰화 + 표제어 추출 => (4) 불용어 제거**

| 단계 | 내용 | 도구 |
|------|------|------|
| **(1) 정제** | 특수문자, 숫자, HTML 태그 제거 | 정규표현식 |
| **(2) 정규화** | 소문자 변환 | `.lower()` |
| **(3) 형태소 분석** | 토큰화 + Lemmatization | NLTK |
| **(4) 불용어 제거** | 영어 기본 + 도메인 불용어 | `CUSTOM_STOPWORDS` |

In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# NLTK 데이터 다운로드 (최초 1회)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

lemmatizer = WordNetLemmatizer()

# 영화 도메인 불용어: 줄거리 서술에 공통적으로 등장하는 일반 표현
MOVIE_STOPWORDS = {
    # 서술 동사 / 일반 동사
    'find', 'finds', 'must', 'gets', 'takes', 'goes', 'make', 'makes',
    'come', 'comes', 'take', 'set', 'sets', 'get', 'let', 'tells',
    'turns', 'begins', 'tries', 'leads', 'wants', 'lives', 'discovers',
    'learns', 'decides', 'becomes', 'returns', 'meets', 'help', 'helps',
    # 영화/스토리 메타 표현
    'film', 'movie', 'story', 'series', 'based', 'tells', 'follows',
    'chronicles', 'centers', 'revolves', 'depicts', 'directed',
    # 일반 명사 (비차별적)
    'life', 'world', 'man', 'woman', 'young', 'new', 'old', 'day',
    'time', 'people', 'way', 'home', 'family', 'father', 'mother',
    'son', 'daughter', 'wife', 'husband', 'girl', 'boy', 'friend',
    'group', 'town', 'city', 'place', 'night', 'years', 'year',
    # 수량/대명사/일반 형용사
    'one', 'two', 'three', 'first', 'last', 'back', 'just', 'like',
    'also', 'however', 'soon', 'later', 'former', 'named',
}

# 영어 기본 불용어 + 도메인 불용어 결합
CUSTOM_STOPWORDS = list(ENGLISH_STOP_WORDS | MOVIE_STOPWORDS)
print(f"불용어 수: sklearn 기본 {len(ENGLISH_STOP_WORDS)}개 + 도메인 {len(MOVIE_STOPWORDS)}개 = 합계 {len(CUSTOM_STOPWORDS)}개")

# TODO 0-3: 영어 텍스트 데이터 전처리 함수 구현
def preprocess_english(text):
    """영어 텍스트 전처리: 정제 => 정규화 => 토큰화 + Lemmatization => 불용어 제거"""
    # (1) 정제: HTML 태그, 특수문자, 숫자 제거
    

    # (2) 정규화: 소문자 변환
    

    # (3) 토큰화 + 표제어 추출 (Lemmatization)
    

    # (4) 불용어 제거 + 2글자 이상 필터링
    tokens = []

    return ' '.join(tokens)


# overview 전처리 적용
print("영어 텍스트 전처리 중...")
df['overview_clean'] = df['overview'].apply(preprocess_english)

# 전처리 전후 비교
print("전처리 완료!")
print()
print("=== 전처리 전후 비교 (첫 번째 문서) ===")
print(f"[원본] {df['overview'].iloc[0][:100]}...")
print(f"[전처리] {df['overview_clean'].iloc[0][:100]}...")
print()

# 전처리 후 텍스트 통계 업데이트
df['word_count'] = df['overview_clean'].str.split().str.len()
df['char_count'] = df['overview_clean'].str.len()
print(f"전처리 후 평균 단어 수: {df['word_count'].mean():.1f}개")

# 분류 모델용 텍스트 (장르 + 전처리된 overview)
df['text'] = df['genres_text'] + ' ' + df['overview_clean']


 ### 0-4. 클래스 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# countplot
sns.countplot(data=df, x='label_name', palette=[COLORS['hit'], COLORS['flop']],
              order=['Hit', 'Flop'], ax=axes[0])
axes[0].set_title('Hit / Flop 분포', fontsize=14)
axes[0].set_xlabel('클래스')
axes[0].set_ylabel('건수')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

# pie chart
counts = df['label_name'].value_counts()
axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=[COLORS['hit'], COLORS['flop']], startangle=90)
axes[1].set_title('Hit / Flop 비율', fontsize=14)

plt.suptitle('클래스 분포 (Hit >= 8.0 / Flop <= 5.0)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print(f"데이터 준비 완료! Part 1부터 분석을 시작하세요.")


---
## Part 1: TF-IDF 분석 + 연관 분석 — "Hit/Flop 영화의 텍스트 특징은?"

### 문제 1-1. Hit/Flop 텍스트 빈도 비교 분석

Hit/Flop 영화의 overview 텍스트를 비교 분석하세요.

- **(a)** Hit/Flop별 word_count, char_count 기술통계 + 히스토그램/박스플롯 비교
- **(b)** TF-IDF 차이(Hit 평균 - Flop 평균) 기반으로 Hit/Flop 각각의 특징 단어 20개를 추출하고 시각화
- **(c)** Bigram(2-gram)으로 Hit/Flop별 상위 20개 빈출 표현을 비교
- **(d)** 워드클라우드로 Hit/Flop 텍스트를 시각적으로 비교
- **(e)** 결과를 종합하여, Hit 영화와 Flop 영화의 텍스트 차이를 **2~3문장으로 해석**

In [ ]:
# 1-1(a): Hit/Flop별 텍스트 길이 기술통계
print("=== Hit/Flop별 word_count 기술통계 ===")
display(df.groupby('label_name')['word_count'].describe())

print()
print("=== Hit/Flop별 char_count 기술통계 ===")
display(df.groupby('label_name')['char_count'].describe())


In [ ]:
# 1-1(a): 히스토그램 + 박스플롯
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# word_count 히스토그램
for label_name, color in [('Hit', COLORS['hit']), ('Flop', COLORS['flop'])]:
    subset = df[df['label_name'] == label_name]
    axes[0, 0].hist(subset['word_count'], bins=30, alpha=0.5, label=label_name, color=color)
axes[0, 0].set_title('단어 수 분포 (Hit vs Flop)', fontsize=12)
axes[0, 0].set_xlabel('단어 수')
axes[0, 0].legend()

# char_count 히스토그램
for label_name, color in [('Hit', COLORS['hit']), ('Flop', COLORS['flop'])]:
    subset = df[df['label_name'] == label_name]
    axes[0, 1].hist(subset['char_count'], bins=30, alpha=0.5, label=label_name, color=color)
axes[0, 1].set_title('글자 수 분포 (Hit vs Flop)', fontsize=12)
axes[0, 1].set_xlabel('글자 수')
axes[0, 1].legend()

# 박스플롯
sns.boxplot(data=df, x='label_name', y='word_count',
            palette=[COLORS['hit'], COLORS['flop']], order=['Hit', 'Flop'], ax=axes[1, 0])
axes[1, 0].set_title('단어 수 박스플롯', fontsize=12)

sns.boxplot(data=df, x='label_name', y='char_count',
            palette=[COLORS['hit'], COLORS['flop']], order=['Hit', 'Flop'], ax=axes[1, 1])
axes[1, 1].set_title('글자 수 박스플롯', fontsize=12)

plt.suptitle('텍스트 길이 비교: Hit vs Flop', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Hit/Flop 인덱스 (이후 분석에서 공통 사용)
hit_idx = df[df['label'] == 1].index.tolist()
flop_idx = df[df['label'] == 0].index.tolist()


In [ ]:
# TODO 1-1(b): TF-IDF 차이 기반 특징 단어 추출 + 시각화
# - TF-IDF 벡터라이저로 overview_clean에 대한 TF-IDF 행렬 생성
# - Hit/Flop별 평균 TF-IDF를 구하고, 차이(Hit - Flop)로 각 그룹의 특징 단어 20개 추출
# - 결과를 DataFrame으로 출력하고 막대그래프로 시각화



In [ ]:
# TODO 1-1(c): Bigram 차이 분석 + 시각화
# - Bigram(2-gram) DTM 생성
# - Hit/Flop별 평균 빈도 차이로 각 그룹의 특징 Bigram 20개 추출
# - 결과를 DataFrame으로 출력하고 막대그래프로 시각화



In [ ]:
# TODO 1-1(d): 워드클라우드
# - TF-IDF 차이 기반으로 Hit/Flop 각각의 고유 키워드로 워드클라우드 생성
# - 1x2 서브플롯으로 Hit/Flop 워드클라우드 비교



In [ ]:
# TODO 1-1(e): 결과 해석
# - 텍스트 길이(word_count) 차이가 있는지?
# - TF-IDF 특징 단어에서 Hit/Flop 각각 어떤 주제가 두드러지는지?
# - Bigram에서 Hit/Flop 각각 어떤 표현 패턴이 나타나는지?



### 문제 1-2. Hit/Flop 동시출현 네트워크 분석

Hit/Flop 영화 각각의 동시출현(Co-occurrence) 네트워크를 분석하세요.

- **(a)** CountVectorizer(binary=True)로 DTM을 생성하고, `DTM.T @ DTM`으로 전체 동시출현 행렬 계산
- **(b)** 전체 동시출현 상위 20쌍을 표와 막대그래프로 시각화
- **(c)** Hit/Flop 각각의 동시출현 네트워크를 pyvis HTML로 생성 (create_network_graph 함수 활용)
- **(d)** Hit/Flop 네트워크의 차이점을 **2~3문장으로 해석**

In [ ]:
# TODO 1-2(a): 동시출현 행렬 생성
# - TF-IDF 차이 기반 고유 키워드를 어휘로 사용하여 CountVectorizer(binary=True) 적용
# - DTM.T @ DTM 으로 동시출현 행렬 계산 (대각선은 0으로 설정)
# - 상위 동시출현 쌍을 리스트로 추출



In [ ]:
# TODO 1-2(b): 상위 20쌍 표 + 막대그래프



In [ ]:
# 1-2(c): Hit/Flop 동시출현 네트워크 (pyvis HTML)
from pyvis.network import Network


def create_network_graph(pairs, filename, title='', top_n=30,
                         node_color='#3498db', height='700px'):
    """
    동시출현 쌍 리스트를 받아서 pyvis 인터랙티브 네트워크 HTML을 생성합니다.
    """
    G = nx.Graph()
    for w1, w2, count in pairs[:top_n]:
        G.add_edge(w1, w2, weight=int(count))

    net = Network(height=height, width='100%', bgcolor='#ffffff', font_color='#333333')

    # 노드 크기 (min-max 정규화)
    node_weights = {}
    for node in G.nodes():
        node_weights[node] = sum(d['weight'] for _, _, d in G.edges(node, data=True))
    max_nw = max(node_weights.values())
    min_nw = min(node_weights.values())
    nw_range = max_nw - min_nw if max_nw != min_nw else 1

    for node in G.nodes():
        normalized_size = 15 + (node_weights[node] - min_nw) / nw_range * 35
        net.add_node(node, label=node, size=normalized_size, color=node_color,
                     font={'size': 16, 'face': 'Arial'})

    # 엣지
    max_ew = max(d['weight'] for _, _, d in G.edges(data=True))
    for u, v, d in G.edges(data=True):
        w = d['weight']
        net.add_edge(u, v, width=1 + (w / max_ew) * 7,
                     color={'color': '#aaaaaa', 'opacity': 0.6},
                     title=f'{u} + {v}: {w}')

    # 물리 엔진
    net.set_options('''
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -80,
          "centralGravity": 0.01,
          "springLength": 150,
          "springConstant": 0.05
        },
        "solver": "forceAtlas2Based",
        "stabilization": {"iterations": 200}
      },
      "interaction": {"hover": true, "zoomView": true, "dragNodes": true}
    }
    ''')

    net.save_graph(filename)

    if title:
        with open(filename, 'r', encoding='utf-8') as f:
            html = f.read()
        title_tag = f'<h2 style="text-align:center; color:#333;">{title}</h2>'
        html = html.replace('<body>', f'<body>\n{title_tag}', 1)
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(html)

    print(f"=> {filename} 저장 완료" + (f" ({title})" if title else ""))



In [ ]:
# TODO 1-2(c): Hit/Flop 각각의 동시출현 네트워크 HTML 생성
# - Hit/Flop 각 그룹별로 동시출현 행렬을 계산하고 쌍을 추출
# - create_network_graph 함수를 활용하여 HTML 파일 생성



In [ ]:
# TODO 1-2(d): 네트워크 해석
# - 전체 동시출현 상위 쌍에서 어떤 표현이 두드러지는지?
# - Hit/Flop 네트워크에서 중심 단어와 연결 패턴이 어떻게 다른지?



---
## Part 2: BERTopic 토픽 모델링 — "영화 줄거리에 숨겨진 토픽은?"

### 문제 2-1. BERTopic으로 영화 줄거리 토픽 분석

SentenceTransformer 임베딩 + BERTopic으로 영화 줄거리의 토픽을 자동 도출하고,
Hit/Flop별 토픽 분포를 분석하세요.

> **참고:** 여기서는 **전체 영화 줄거리(Hit + Flop 통합)** 를 대상으로 BERTopic을 수행합니다. \
> 전체 코퍼스에서 토픽을 도출한 뒤, 각 토픽이 Hit/Flop과 어떤 관계가 있는지를 사후 분석하는 방식입니다. \
> 추가 실습으로 **Hit 영화만, Flop 영화만 따로 BERTopic을 적용** 하여 각 그룹에서 어떤 토픽이 나오는지 비교해 보세요.

- **(a)** SentenceTransformer로 overview 임베딩 생성 + UMAP 2D 산점도 (Hit/Flop 색상 구분)
- **(b)** BERTopic 모델 구성 (UMAP, HDBSCAN, CountVectorizer 직접 설정) 및 학습
- **(c)** BERTopic내의 하이퍼 파라미터 적절한 설정을 통한 토픽 탐색
- **(d)** KeyBERTInspired로 키워드 품질 개선 + 토픽 요약 테이블 확인
- **(e)** BERTopic 내장 시각화 5종 수행: topics, barchart, heatmap, hierarchy, documents
- **(f)** Hit/Flop별 토픽 분포 분석: 토픽별 Hit 비율 + 토픽-흥행 연결 종합 테이블
- **(g)** 분석 결과를 바탕으로 **어떤 토픽의 영화가 흥행하는지 2~3문장으로 해석**

In [ ]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

# TODO 2-1(a): 임베딩 생성 + UMAP 2D 산점도
#
# [1] SentenceTransformer 모델 로드 후 df['overview']를 임베딩
#   - 캐싱: np.save/np.load로 임베딩을 파일에 저장/로드하면 재실행 시 빠름
# [2] UMAP(n_components=2)로 2D 차원 축소
# [3] Hit/Flop 색상 구분 산점도 시각화

embeddings = None  # np.ndarray, shape: (n_docs, embedding_dim), SentenceTransformer 문서 임베딩 벡터


In [ ]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

# TODO 2-1(b): BERTopic 모델 구성 및 학습
#
# [1] 서브 모델 설정
#   - UMAP(n_components=5, ...) : 차원 축소 (시각화용 2D와 별도)
#   - HDBSCAN(min_cluster_size=...) : 밀도 기반 클러스터링
#   - CountVectorizer(stop_words=CUSTOM_STOPWORDS, ngram_range=(1,2), ...) : 토픽 키워드 추출
#   - KeyBERTInspired() : 키워드 품질 개선
# [2] BERTopic(...) 모델 생성 후 fit_transform(docs, embeddings=embeddings) 학습

In [ ]:
# TODO 2-1(c): 토픽 요약 테이블 확인
# - 토픽 정보 테이블 출력
# - 주요 토픽별 대표 키워드와 대표 문서 확인



In [ ]:
# TODO 2-1(e): BERTopic 내장 시각화 5종 + reduce_topics
#
# [1] visualize_topics()         : 토픽 간 거리 맵 (원 크기 = 문서 수)
# [2] visualize_barchart()       : 토픽별 키워드 막대 차트
# [3] visualize_heatmap()        : 토픽 유사도 히트맵
# [4] visualize_hierarchy()      : 토픽 계층 구조 (덴드로그램)
# [5] 위 히트맵/덴드로그램을 참고하여 reduce_topics(docs, nr_topics=N)로 유사 토픽 병합
# [6] 병합 후 visualize_documents()로 문서-토픽 분포 확인


In [ ]:
# TODO 2-1(f): Hit/Flop별 토픽 분포 분석
#
# [1] df['topic'] = topic_model.topics_ 로 토픽 번호 할당 (노이즈 -1 제외)
# [2] groupby('topic')로 종합 테이블 생성:
#   - count, hit_count, flop_count, hit_rate, avg_vote, avg_popularity 등
# [3] 시각화 (1x2 서브플롯):
#   - 토픽별 Hit 비율 막대그래프 (0.5 기준선 포함)
#   - 토픽별 Hit/Flop 건수 누적 막대그래프


In [ ]:
# TODO 2-1(g): 결과 해석
# - 몇 개의 토픽이 도출되었고, 토픽별 Hit 비율 범위는 어떠한지?
# - Hit 비율이 가장 높은/낮은 토픽의 키워드는 무엇인지?
# - 토픽별로 어떤 장르/주제가 흥행과 연결되는지?


---
## Part 3: 텍스트 분류 모델 — "텍스트만으로 흥행 여부를 예측할 수 있는가?"

### 문제 3-1. BERT 파인튜닝으로 Hit/Flop 분류 모델 학습 및 평가

HuggingFace Transformers의 `AutoModelForSequenceClassification`과 `Trainer`를 사용하여
영화 텍스트(genres + overview)로 Hit/Flop을 분류하는 모델을 학습하고 평가하세요.
> 딥러닝 모델의 빠른 학습을 위해선 gpu가 필요하고, gpu 버전 torch 라이브러리를 설치해야합니다.

- **(a)** train/test split (stratify) + HuggingFace Dataset 토큰화
- **(b)** BERT 파인튜닝 학습 (from_pretrained + Trainer)
- **(c)** classification_report + confusion matrix 히트맵
- **(d)** ROC 곡선 + AUC 계산
- **(e)** 오분류 분석: False Positive / False Negative 상위 10건 확인 + 장르별 오분류율
- **(f)** 분류 모델의 성능과 한계를 **2~3문장으로 해석**

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments
)
from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# TODO 3-1(a): 데이터 준비
#
# [1] train/test split (test_size=0.2, stratify=df['label'], random_state=42)
#   - df['text']를 입력, df['label']을 타깃으로 분리
#
# [2] Undersampling으로 클래스 불균형 해소
#   - 학습 데이터에서 다수 클래스(Flop)를 소수 클래스(Hit) 수에 맞춰 축소
#   - 테스트 데이터는 원본 비율 유지 (변경하지 않음)
#
# [3] HuggingFace Dataset 토큰화
#   - HFDataset.from_dict()으로 변환 후 tokenizer로 토큰화
#   - set_format("torch")로 PyTorch 텐서 형식 설정

In [ ]:
# TODO 3-1(b): BERT 파인튜닝 학습
#
# [1] 모델 로드: AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# [2] TrainingArguments 설정 (output_dir, epochs, batch_size, eval_strategy 등)
# [3] Trainer(model, args, train_dataset, eval_dataset, compute_metrics) 로 학습
# [4] 학습 후 eval_results에서 f1_val 추출

model = None    # AutoModelForSequenceClassification, BERT 분류 모델
trainer = None  # Trainer, HuggingFace 학습 관리자
f1_val = None   # float, BERT 파인튜닝 모델의 weighted F1-Score


In [ ]:
# TODO 3-1(c): classification_report + confusion matrix 히트맵
#
# [1] trainer.predict(test_dataset)로 예측 수행
# [2] classification_report 출력 (target_names=['Flop', 'Hit'])
# [3] confusion_matrix 히트맵 시각화
# [4] 학습/테스트 정확도 계산하여 train_acc, test_acc에 저장

train_acc = None  # float, BERT 파인튜닝 모델의 학습 정확도
test_acc = None   # float, BERT 파인튜닝 모델의 테스트 정확도


In [ ]:
from sklearn.metrics import roc_curve, auc
import torch

# TODO 3-1(d): ROC 곡선 + AUC 계산
#
# [1] logits => softmax => Hit(1) 확률 추출 (y_score)
# [2] roc_curve(y_true, y_score)로 fpr, tpr 계산
# [3] auc(fpr, tpr)로 AUC 계산
# [4] ROC 곡선 시각화 (대각선 기준선 포함)

fpr = None      # np.ndarray, BERT 모델의 위양성률 (False Positive Rate)
tpr = None      # np.ndarray, BERT 모델의 진양성률 (True Positive Rate)
roc_auc = None  # float, BERT 모델의 AUC 값


In [ ]:
# TODO 3-1(e): 오분류 분석
#
# [1] 테스트 데이터 DataFrame 구성
#   - train_test_split으로 테스트 인덱스를 다시 추출하여
#     title, vote_average, genres_text, y_true, y_pred, prob_hit 열을 가진 df_test 생성
#
# [2] 오분류 샘플 분류
#   - False Positive (FP): 실제 Flop인데 Hit으로 예측 (y_true=0, y_pred=1)
#   - False Negative (FN): 실제 Hit인데 Flop으로 예측 (y_true=1, y_pred=0)
#   - 각각 prob_hit 기준으로 정렬하여 상위 10건 출력
#
# 예시 출력:
# | title              | vote_average | genres_text      | prob_hit |
# |--------------------|-------------|------------------|----------|
# | Movie A            | 4.2         | Action Thriller  | 0.9123   |
#
# [3] 장르별 오분류율
#   - 주요 장르 10개에 대해 해당 장르 포함 영화의 오분류율 계산
#
# | genre    | count | error_rate |
# |----------|-------|------------|
# | Horror   | 85    | 0.4235     |


In [ ]:
# TODO 3-1(f): 결과 해석
# - 테스트 정확도와 AUC는 어느 수준인지?
# - 오분류된 영화들의 공통적인 특징은 무엇인지?
# - 텍스트만으로 예측의 한계는 무엇이고, 어떤 정보를 추가하면 개선될 수 있는지?

# 로드된 모델 메모리 정리
del model, trainer
gc.collect()


### 문제 3-2. 로지스틱 회귀 비교 모델 (TF-IDF / SentenceTransformer 임베딩)

TF-IDF 벡터와 SentenceTransformer 임베딩을 각각 로지스틱 회귀에 넣어 학습하고,
BERT 파인튜닝 모델과 3종 성능을 비교하세요.

- **(a)** TF-IDF 임베딩 + 로지스틱 회귀 학습 및 평가
- **(b)** SentenceTransformer 임베딩 + 로지스틱 회귀 학습 및 평가
- **(c)** 3종 모델 ROC 곡선 비교
- **(d)** 3종 모델 성능 비교 테이블 + 해석

In [ ]:
# 3-2(a): TF-IDF 임베딩 + 로지스틱 회귀
from sklearn.linear_model import LogisticRegression

# Part 1에서 사용한 것과 동일한 설정의 TF-IDF 벡터라이저
tfidf_clf = TfidfVectorizer(
    stop_words=CUSTOM_STOPWORDS,
    min_df=2, max_df=0.85, max_features=5000
)
tfidf_all = tfidf_clf.fit_transform(df['overview_clean'])
print(f"TF-IDF 행렬 shape: {tfidf_all.shape}")

# train/test split (BERT와 동일한 random_state, test_size)
X_train_tfidf, X_test_tfidf, y_train_lr, y_test_lr = train_test_split(
    tfidf_all, df['label'].values,
    test_size=0.2, random_state=42, stratify=df['label'].values
)

# 로지스틱 회귀 학습 (TF-IDF는 이미 정규화되어 있으므로 스케일링 불필요)
lr_tfidf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr_tfidf.fit(X_train_tfidf, y_train_lr)

# 예측
y_pred_tfidf = lr_tfidf.predict(X_test_tfidf)
y_score_tfidf = lr_tfidf.predict_proba(X_test_tfidf)[:, 1]

tfidf_train_acc = lr_tfidf.score(X_train_tfidf, y_train_lr)
tfidf_test_acc = accuracy_score(y_test_lr, y_pred_tfidf)
tfidf_f1 = f1_score(y_test_lr, y_pred_tfidf, average='weighted')

print()
print("[TF-IDF + 로지스틱 회귀]")
print(f"  학습 정확도: {tfidf_train_acc:.4f} ({tfidf_train_acc*100:.1f}%)")
print(f"  테스트 정확도: {tfidf_test_acc:.4f} ({tfidf_test_acc*100:.1f}%)")
print(f"  F1-Score (weighted): {tfidf_f1:.4f}")


In [ ]:
print("=== Classification Report (TF-IDF + Logistic Regression) ===")
print(classification_report(y_test_lr, y_pred_tfidf, target_names=['Flop', 'Hit']))

# Confusion Matrix
cm_tfidf = confusion_matrix(y_test_lr, y_pred_tfidf)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_tfidf, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Flop', 'Hit'], yticklabels=['Flop', 'Hit'], ax=ax)
ax.set_xlabel('예측값', fontsize=12)
ax.set_ylabel('실제값', fontsize=12)
ax.set_title('혼동 행렬 (TF-IDF + 로지스틱 회귀)', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# 3-2(b): SentenceTransformer 임베딩 + 로지스틱 회귀

# Part 2에서 생성한 embeddings 재활용 (SentenceTransformer 임베딩)
print(f"SentenceTransformer 임베딩 shape: {embeddings.shape}")

# train/test split (동일한 random_state, test_size)
X_train_emb, X_test_emb, _, _ = train_test_split(
    embeddings, df['label'].values,
    test_size=0.2, random_state=42, stratify=df['label'].values
)

# 로지스틱 회귀 학습
lr_emb = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr_emb.fit(X_train_emb, y_train_lr)

# 예측
y_pred_emb = lr_emb.predict(X_test_emb)
y_score_emb = lr_emb.predict_proba(X_test_emb)[:, 1]

emb_train_acc = lr_emb.score(X_train_emb, y_train_lr)
emb_test_acc = accuracy_score(y_test_lr, y_pred_emb)
emb_f1 = f1_score(y_test_lr, y_pred_emb, average='weighted')

print()
print("[SentenceTransformer + 로지스틱 회귀]")
print(f"  학습 정확도: {emb_train_acc:.4f} ({emb_train_acc*100:.1f}%)")
print(f"  테스트 정확도: {emb_test_acc:.4f} ({emb_test_acc*100:.1f}%)")
print(f"  F1-Score (weighted): {emb_f1:.4f}")


In [ ]:
print("=== Classification Report (SentenceTransformer + Logistic Regression) ===")
print(classification_report(y_test_lr, y_pred_emb, target_names=['Flop', 'Hit']))

# Confusion Matrix
cm_emb = confusion_matrix(y_test_lr, y_pred_emb)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_emb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Flop', 'Hit'], yticklabels=['Flop', 'Hit'], ax=ax)
ax.set_xlabel('예측값', fontsize=12)
ax.set_ylabel('실제값', fontsize=12)
ax.set_title('혼동 행렬 (SentenceTransformer + 로지스틱 회귀)', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# 3-2(c): 3종 모델 ROC 곡선 비교
from sklearn.metrics import roc_curve, auc

fpr_tfidf, tpr_tfidf, _ = roc_curve(y_test_lr, y_score_tfidf)
roc_auc_tfidf = auc(fpr_tfidf, tpr_tfidf)

fpr_emb, tpr_emb, _ = roc_curve(y_test_lr, y_score_emb)
roc_auc_emb = auc(fpr_emb, tpr_emb)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color=COLORS['blue'], lw=2,
        label=f'ELECTRA Fine-tuned (AUC = {roc_auc:.4f})')
ax.plot(fpr_tfidf, tpr_tfidf, color=COLORS['violet'], lw=2,
        label=f'TF-IDF + LogReg (AUC = {roc_auc_tfidf:.4f})')
ax.plot(fpr_emb, tpr_emb, color=COLORS['amber'], lw=2,
        label=f'SentenceTransformer + LogReg (AUC = {roc_auc_emb:.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
ax.set_xlabel('위양성률 (FPR)', fontsize=12)
ax.set_ylabel('진양성률 (TPR)', fontsize=12)
ax.set_title('ROC 곡선 비교: 3종 모델', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
plt.tight_layout()
plt.show()


In [ ]:
# 3-2(d): 3종 모델 성능 비교 테이블 + 해석
comparison_df = pd.DataFrame({
    '모델': [
        'ELECTRA (Fine-tuned)',
        'TF-IDF + Logistic Regression',
        'SentenceTransformer + Logistic Regression'
    ],
    '입력 표현': [
        'WordPiece 토큰 (128 토큰)',
        f'TF-IDF 벡터 ({tfidf_all.shape[1]:,}차원)',
        f'GIST 임베딩 ({embeddings.shape[1]}차원)'
    ],
    '학습 정확도': [f"{train_acc:.4f}", f"{tfidf_train_acc:.4f}", f"{emb_train_acc:.4f}"],
    '테스트 정확도': [f"{test_acc:.4f}", f"{tfidf_test_acc:.4f}", f"{emb_test_acc:.4f}"],
    'F1-Score': [f"{f1_val:.4f}", f"{tfidf_f1:.4f}", f"{emb_f1:.4f}"],
    'AUC': [f"{roc_auc:.4f}", f"{roc_auc_tfidf:.4f}", f"{roc_auc_emb:.4f}"],
})
print("=== 3종 모델 성능 비교 ===")
display(comparison_df)

print()
print("=== 비교 분석 결과 해석 ===")
print()
print("여기에 분석 결과를 바탕으로 해석을 작성하세요.")
print("- 3종 모델 중 어떤 모델이 가장 높은 성능을 보이는지?")
print("- TF-IDF(희소 표현), SentenceTransformer(밀집 표현), BERT(파인튜닝) 각각의 특징은?")
print("- 텍스트만으로 예측의 한계와 개선 방향은?")


---
## 수고하셨습니다!